In [ ]:
## S6: Baseline models
# Archived notebook. Main thesis flow now uses 04_extension_smote_xgboost.ipynb.
# Original run order was: 01_eda.ipynb -> 02_feature_engineering.ipynb -> 03_prepare_dataset.ipynb -> 04_baseline_models.ipynb

from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import time
import sys

processed_path = Path('../data/processed_split.pkl')
if not processed_path.exists():
    raise FileNotFoundError(
        'Missing ../data/processed_split.pkl. Run 03_prepare_dataset.ipynb first.'
    )

sys.path.append('../src')
from evaluation import evaluate_model, print_metrics_table

data = joblib.load(processed_path)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
X_train_scaled, X_test_scaled = data['X_train_scaled'], data['X_test_scaled']

print(f'Du lieu da load: X_train {X_train.shape}, X_test {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.4f}, Test fraud rate: {y_test.mean():.4f}')


In [ ]:
from sklearn.linear_model import LogisticRegression

start = time.time()
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
lr_train_time = time.time() - start
print(f'Thoi gian train LR: {lr_train_time:.2f} giay')

lr_results = evaluate_model(lr, X_test_scaled, y_test, 'Logistic Regression', lr_train_time)
print(lr_results)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

start = time.time()
knn = KNeighborsClassifier(n_neighbors=5, algorithm='auto', n_jobs=-1)
knn.fit(X_train_scaled, y_train)
knn_train_time = time.time() - start
print(f'Thoi gian train KNN: {knn_train_time:.2f} giay')

knn_results = evaluate_model(knn, X_test_scaled, y_test, 'KNN', knn_train_time)
print(knn_results)


In [ ]:
results = print_metrics_table([lr_results, knn_results])
print(results)

from evaluation import compare_with_anchor_paper
comparison = compare_with_anchor_paper(results)
print('\nSo sanh voi anchor paper:')
print(comparison)
